# Notebook 03 — DiCE Counterfactual Explanations
### XAI for FPG Prediction: Algorithmic Fairness in Health Insurance Underwriting
**Paper §3.4, §4.5**

This notebook generates diverse counterfactual explanations (DiCE) for non-normal
FPG cases, quantifies counterfactual quality across six demographic groups, and
prepares structured outputs for the LLM report pipeline (Notebook 04).

**Two transition pathways:**
- Diabetes (FPG ≥ 126) → IFG (100–125 mg/dL)
- IFG (100–125) → Normal (75–99.9 mg/dL)

**Quality metrics:** Proximity · Diversity · Feasibility · Sparsity

> **Prerequisite**: Run `01_regression_models.ipynb` first.


## 1. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import dice_ml
from dice_ml import Dice
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.unicode_minus": False,
    "figure.dpi":         150,
})

SEED       = 42
DPI        = 300
N_CF       = 3    # counterfactuals per case
N_CASES    = 3    # representative cases per group × transition
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)

# ── Group configuration ───────────────────────────────────────────────────────
GROUP_CONFIG = {
    "Young_Male":    {"age_group": 0.0, "sex_code": 1.0},
    "Young_Female":  {"age_group": 0.0, "sex_code": 2.0},
    "Middle_Male":   {"age_group": 1.0, "sex_code": 1.0},
    "Middle_Female": {"age_group": 1.0, "sex_code": 2.0},
    "Elderly_Male":  {"age_group": 2.0, "sex_code": 1.0},
    "Elderly_Female":{"age_group": 2.0, "sex_code": 2.0},
}
GROUP_LABELS = {
    "Young_Male":    "Young Male",       "Young_Female":  "Young Female",
    "Middle_Male":   "Middle-aged Male", "Middle_Female": "Middle-aged Female",
    "Elderly_Male":  "Elderly Male",     "Elderly_Female":"Elderly Female",
}
GROUP_COLORS = {
    "Young_Male":    "#1565C0", "Young_Female":  "#90CAF9",
    "Middle_Male":   "#E65100", "Middle_Female": "#FFCC80",
    "Elderly_Male":  "#2E7D32", "Elderly_Female":"#A5D6A7",
}

# ── Transition targets ────────────────────────────────────────────────────────
TRANSITIONS = {
    "diabetes": {"target_range": (100.0, 125.0),
                 "label": "Diabetes → IFG",
                 "source_label": "Diabetes (≥126)",
                 "target_label": "IFG (100–125)"},
    "ifg":      {"target_range": (75.0,  99.9),
                 "label": "IFG → Normal",
                 "source_label": "IFG (100–125)",
                 "target_label": "Normal (<100)"},
}

def assign_glucose_stage(fpg: float) -> str:
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"

log.info("Setup complete.")


2026-04-27 15:14:41,751 | INFO | Setup complete.


## 2. Load Artefacts from Notebook 01

In [2]:
df_final          = pd.read_parquet(OUTPUT_DIR / "df_final.parquet")
best_models       = joblib.load(OUTPUT_DIR / "best_models.pkl")
best_algo_name    = joblib.load(OUTPUT_DIR / "best_algo_name.pkl")
best_params_store = joblib.load(OUTPUT_DIR / "best_params_store.pkl")

NUM_FEATURES = [
    "BMI", "WaistCirc", "Weight",
    "Energy_kcal", "Carb_g",  "Sugar_g",  "Sodium_mg",
    "Fat_g",       "SatFat_g","Fiber_g",   "Potassium_mg", "Protein_g",
]
CAT_FEATURES = [
    "ObesityStatus",    "WeightChangeStatus", "WeightLossAmount", "WeightGainAmount",
    "DrinkingFrequency","DrinkingAmount",      "SmokingStatus",
    "VigorousAct_Work", "VigorousAct_Leisure","ModerateAct_Work",
    "WalkingActivity",  "AerobicRate",         "BreakfastFreq",
    "StressLevel",      "StressAwareness",
    "IncomeQuartile",   "HouseholdIncome",     "EducationLevel", "HealthScreening",
]
X_FEATURES = NUM_FEATURES + CAT_FEATURES

log.info("Artefacts loaded. df_final: %s", df_final.shape)


2026-04-27 15:14:42,293 | INFO | Artefacts loaded. df_final: (16677, 36)


## 3. DiCE Helper Functions

In [3]:
def force_float(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """Cast all feature columns to float64 (required by DiCE)."""
    df = df.copy()
    for col in features:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    return df


def build_dice_explainer(model, df_train: pd.DataFrame,
                         feature_names: list, outcome_name: str = "FPG"):
    """
    Initialise DiCE Data + Model objects for regression.
    df_train must have float64 features and the outcome column.
    """
    d   = dice_ml.Data(
        dataframe           = df_train[feature_names + [outcome_name]],
        continuous_features = feature_names,
        outcome_name        = outcome_name,
    )
    m   = dice_ml.Model(model=model, backend="sklearn", model_type="regressor")
    exp = Dice(d, m, method="random")
    return exp


def generate_counterfactuals(explainer, query: pd.DataFrame,
                             target_range: tuple, n_cf: int = 3):
    """
    Generate n_cf counterfactual instances for a single query.

    Parameters
    ----------
    query        : float64 DataFrame, shape (1, n_features)
    target_range : (low, high) desired FPG range
    Returns None on failure.
    """
    query = force_float(query, query.columns.tolist())
    try:
        result = explainer.generate_counterfactuals(
            query_instances  = query,
            total_CFs        = n_cf,
            desired_range    = list(target_range),
            permitted_range  = None,
            features_to_vary = "all",
            random_seed      = SEED,
        )
        return result
    except Exception as exc:
        log.warning("CF generation failed: %s", exc)
        return None


# ── Quality metrics ───────────────────────────────────────────────────────────
def _feature_stds(df_train: pd.DataFrame, features: list) -> np.ndarray:
    """Per-feature standard deviations from training data (used for normalisation)."""
    stds = df_train[features].std().values
    stds[stds == 0] = 1.0
    return stds


def proximity(original: pd.DataFrame, cf_df: pd.DataFrame,
              features: list, stds: np.ndarray) -> float:
    """
    Mean normalised L1 distance from original to each counterfactual.
    Lower is better — indicates minimal lifestyle change required.
    """
    orig   = original[features].values.flatten().astype(float)
    scores = [float(np.mean(np.abs(orig - row.values.astype(float)) / stds))
              for _, row in cf_df[features].iterrows()]
    return round(float(np.mean(scores)), 4)


def diversity(cf_df: pd.DataFrame, features: list, stds: np.ndarray) -> float:
    """
    Mean pairwise normalised L1 distance among counterfactuals.
    Higher is better — more diverse action pathways offered.
    """
    if len(cf_df) < 2:
        return 0.0
    vals  = cf_df[features].values.astype(float)
    dists = [float(np.mean(np.abs(vals[i] - vals[j]) / stds))
             for i in range(len(vals)) for j in range(i+1, len(vals))]
    return round(float(np.mean(dists)), 4)


def feasibility(original: pd.DataFrame, cf_df: pd.DataFrame,
                features: list, cat_features: list) -> float:
    """
    Proportion of categorical feature changes ≤ 1 unit.
    Higher is better — changes must be clinically realistic.
    """
    orig_cats = original[cat_features].values.flatten().astype(float)
    n_total   = len(cf_df) * len(cat_features)
    if n_total == 0:
        return 0.0
    n_valid = sum(
        int(abs(cv - ov) <= 1.0)
        for _, row in cf_df[cat_features].iterrows()
        for ov, cv in zip(orig_cats, row.values.astype(float))
    )
    return round(n_valid / n_total, 4)


def sparsity(original: pd.DataFrame, cf_df: pd.DataFrame,
             features: list, tol: float = 1e-3) -> float:
    """
    Mean proportion of features changed relative to the original.
    Lower is better — fewer changes → more actionable recommendation.
    """
    orig   = original[features].values.flatten().astype(float)
    scores = [np.sum(np.abs(orig - row.values.astype(float)) > tol) / len(features)
              for _, row in cf_df[features].iterrows()]
    return round(float(np.mean(scores)), 4)


log.info("DiCE utilities defined.")


2026-04-27 15:14:42,343 | INFO | DiCE utilities defined.


## 4. Stratified DiCE Generation

In [4]:
log.info("=" * 60)
log.info("DiCE counterfactual generation — 6 groups × 2 transitions")
log.info("=" * 60)

all_cf_results  = {}   # {group: {transition: [case_dicts]}}
quality_records = []   # flat list for quality DataFrame

for grp, cfg in GROUP_CONFIG.items():
    log.info("── Group: %s ──", GROUP_LABELS[grp])
    all_cf_results[grp] = {}

    df_g = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy().reset_index(drop=True)

    df_g = force_float(df_g, X_FEATURES)
    df_g["FPG"] = pd.to_numeric(df_g["FPG"], errors="coerce").astype("float64")

    model = best_models.get(grp)
    if model is None:
        log.warning("  No model for %s — skipping", grp)
        continue

    X = df_g[X_FEATURES]
    y = df_g["FPG"]
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, random_state=SEED)

    df_train = force_float(X_train.copy(), X_FEATURES)
    df_train["FPG"] = y_train.values.astype("float64")
    stds = _feature_stds(df_train, X_FEATURES)

    try:
        explainer = build_dice_explainer(model, df_train, X_FEATURES)
    except Exception as exc:
        log.error("  DiCE explainer failed for %s: %s", grp, exc)
        continue

    df_val = force_float(X_val.copy(), X_FEATURES)
    df_val["FPG"]   = y_val.values.astype("float64")
    df_val["Stage"] = df_val["FPG"].apply(assign_glucose_stage)

    for stage, t_info in TRANSITIONS.items():
        target_range = t_info["target_range"]
        sub = df_val[df_val["Stage"] == stage].reset_index(drop=True)
        if len(sub) == 0:
            log.info("  [%s] No cases in hold-out for stage '%s'", grp, stage)
            continue

        # Select N_CASES cases closest to the group median FPG
        median_fpg  = sub["FPG"].median()
        rep_cases   = sub.iloc[(sub["FPG"] - median_fpg).abs().argsort()].head(N_CASES)
        n_available = len(sub)
        log.info("  [%s → %s]  eligible=%d  representative=%d",
                 t_info["source_label"], t_info["target_label"],
                 n_available, len(rep_cases))

        case_list = []
        for case_idx, (_, row) in enumerate(rep_cases.iterrows()):
            query     = force_float(row[X_FEATURES].to_frame().T.reset_index(drop=True),
                                    X_FEATURES)
            orig_fpg  = float(row["FPG"])

            cf_result = generate_counterfactuals(explainer, query,
                                                 target_range, n_cf=N_CF)
            if cf_result is None:
                continue
            try:
                cf_df = cf_result.cf_examples_list[0].final_cfs_df
                if cf_df is None or len(cf_df) == 0:
                    continue
                cf_df    = force_float(cf_df, X_FEATURES)
                delta_df = cf_df[X_FEATURES].subtract(query[X_FEATURES].values[0],
                                                       axis=1)
                prox = proximity(query, cf_df, X_FEATURES, stds)
                div  = diversity(cf_df, X_FEATURES, stds)
                feas = feasibility(query, cf_df, X_FEATURES, CAT_FEATURES)
                spar = sparsity(query, cf_df, X_FEATURES)
                pred_fpg = model.predict(cf_df[X_FEATURES]).tolist()

                log.info("    Case %d | orig=%.1f  Prox=%.4f  Div=%.4f  "
                         "Feas=%.4f  Spar=%.4f | CF preds: %s",
                         case_idx+1, orig_fpg, prox, div, feas, spar,
                         [f"{p:.1f}" for p in pred_fpg])

                case_list.append({
                    "case_idx": case_idx + 1,
                    "orig_fpg": orig_fpg,
                    "pred_fpg": pred_fpg,
                    "cf_df":    cf_df,
                    "delta_df": delta_df,
                    "query":    query,
                    "n_eligible": n_available,
                })
                quality_records.append({
                    "Group":        GROUP_LABELS[grp],
                    "Transition":   t_info["label"],
                    "Source_Stage": t_info["source_label"],
                    "Target_Stage": t_info["target_label"],
                    "Case":         case_idx + 1,
                    "Orig_FPG":     round(orig_fpg, 2),
                    "CF_FPG_mean":  round(float(np.mean(pred_fpg)), 2),
                    "Proximity":    prox,
                    "Diversity":    div,
                    "Feasibility":  feas,
                    "Sparsity":     spar,
                    "N_eligible":   n_available,
                })
            except Exception as exc:
                log.warning("    Case %d processing error: %s", case_idx+1, exc)

        all_cf_results[grp][stage] = case_list

# Save
joblib.dump(all_cf_results, OUTPUT_DIR / "dice_results.pkl")
quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(OUTPUT_DIR / "dice_quality_metrics.csv", index=False, encoding="utf-8")
log.info("Saved: dice_results.pkl + dice_quality_metrics.csv (%d records)", len(quality_df))


2026-04-27 15:14:42,385 | INFO | ============================================================
2026-04-27 15:14:42,387 | INFO | DiCE counterfactual generation — 6 groups × 2 transitions
2026-04-27 15:14:42,394 | INFO | ============================================================
2026-04-27 15:14:42,398 | INFO | ── Group: Young Male ──
2026-04-27 15:14:42,563 | INFO |   [Diabetes (≥126) → IFG (100–125)]  eligible=12  representative=3
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.35it/s]
2026-04-27 15:14:43,069 | INFO |     Case 1 | orig=133.0  Prox=0.1548  Div=0.3095  Feas=0.9649  Spar=0.0538 | CF preds: ['108.5', '102.6', '103.7']
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.50it/s]
2026-04-27 15:14:43,561 | INFO |     Case 2 | orig=144.0  Prox=0.1463  Div=0.1623  Feas=0.9649  Spar=0.0538 | CF preds: ['100.7', '103.1', '100.7']
100%|███████████████████████

## 5. Quality Metrics Summary

In [5]:
print("── DiCE Quality Metrics Summary ──")
summary = (quality_df
    .groupby(["Transition"])[["Proximity","Diversity","Feasibility","Sparsity"]]
    .agg(["mean","std"])
    .round(4))
print(summary.to_string())

print("\n── Per-Group: IFG → Normal Proximity (key fairness indicator) ──")
prox_ifg = (quality_df[quality_df["Transition"] == "IFG → Normal"]
            .groupby("Group")["Proximity"]
            .agg(["mean","min","max"])
            .round(4))
print(prox_ifg.sort_values("mean", ascending=False).to_string())

print("\n[Metric Interpretation]")
for m, desc in [
    ("Proximity",    "↓ Lower = smaller feature change needed (easier to act on)"),
    ("Diversity",    "↑ Higher = more varied action pathways offered"),
    ("Feasibility",  "↑ Higher = changes within realistic bounds"),
    ("Sparsity",     "↓ Lower = fewer features need to change"),
]:
    print(f"  {m:12s}: {desc}")


── DiCE Quality Metrics Summary ──
               Proximity         Diversity         Feasibility         Sparsity        
                    mean     std      mean     std        mean     std     mean     std
Transition                                                                             
Diabetes → IFG    0.1149  0.0477    0.1937  0.0895      0.9737  0.0183   0.0532  0.0101
IFG → Normal      0.3620  0.3600    0.2773  0.2024      0.9259  0.0793   0.1493  0.1278

── Per-Group: IFG → Normal Proximity (key fairness indicator) ──
                      mean     min     max
Group                                     
Elderly Male        0.9735  0.6872  1.3746
Elderly Female      0.4682  0.1393  0.7765
Middle-aged Male    0.3809  0.2404  0.4522
Young Female        0.1430  0.0977  0.2079
Middle-aged Female  0.1033  0.0759  0.1413
Young Male          0.1029  0.0806  0.1420

[Metric Interpretation]
  Proximity   : ↓ Lower = smaller feature change needed (easier to act on)
  Diversity   :

## 6. LLM-Ready Structured Output (→ Notebook 04)

In [6]:
llm_records = []

for grp, cfg in GROUP_CONFIG.items():
    for stage, t_info in TRANSITIONS.items():
        case_list = all_cf_results.get(grp, {}).get(stage, [])
        for case in case_list:
            delta     = case["delta_df"].mean()
            top5_feat = delta.abs().nlargest(5)
            changes   = {f: round(float(delta[f]), 3) for f in top5_feat.index}

            for i, pred_fpg in enumerate(case["pred_fpg"]):
                # Pseudonymised ID encoding sex (M/F), age group (Y/M/S), stage, case, CF
                sex_code  = "M" if cfg["sex_code"] == 1.0 else "F"
                age_code  = {0.0:"Y", 1.0:"M", 2.0:"S"}[cfg["age_group"]]
                pseudo_id = (f"{sex_code}-{age_code}-XX-"
                             f"{case['case_idx']:04d}-CF{i+1}")

                llm_records.append({
                    "pseudo_id":      pseudo_id,
                    "group":          GROUP_LABELS[grp],
                    "source_stage":   t_info["source_label"],
                    "target_stage":   t_info["target_label"],
                    "case_idx":       case["case_idx"],
                    "cf_idx":         i + 1,
                    "orig_fpg":       round(case["orig_fpg"], 2),
                    "pred_fpg":       round(float(pred_fpg), 2),
                    "fpg_reduction":  round(case["orig_fpg"] - float(pred_fpg), 2),
                    "top5_changes":   str(changes),
                    "proximity":      quality_df.loc[
                        (quality_df["Group"]      == GROUP_LABELS[grp]) &
                        (quality_df["Transition"] == t_info["label"]) &
                        (quality_df["Case"]       == case["case_idx"]),
                        "Proximity"].values[0] if len(quality_df) > 0 else None,
                })

llm_df = pd.DataFrame(llm_records)
llm_df.to_csv(OUTPUT_DIR / "dice_llm_input.csv", index=False, encoding="utf-8")
log.info("LLM input saved: %d rows × %d cols", *llm_df.shape)
print(llm_df[["pseudo_id","group","source_stage","orig_fpg","pred_fpg","top5_changes"]]
      .head(9).to_string(index=False))


2026-04-27 15:15:11,677 | INFO | LLM input saved: 108 rows × 11 cols


      pseudo_id      group    source_stage  orig_fpg  pred_fpg                                                                                                        top5_changes
M-Y-XX-0001-CF1 Young Male Diabetes (≥126)     133.0    108.49 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0001-CF2 Young Male Diabetes (≥126)     133.0    102.58 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0001-CF3 Young Male Diabetes (≥126)     133.0    103.68 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0002-CF1 Young Male Diabetes (≥126)     144.0    100.69                     {'Potassium_mg': 4343.256, 'WaistCirc': 5.1, 'ObesityStatus': 1.033, 'BMI': 0.0, 'Weight': 0.0}
M-Y-XX-0002-CF2 Young Male Diabetes (≥126)     144.0    103.08                     {'Potassium_mg': 4343.

## 7. Figures

In [7]:
# ── Figure: Mean feature changes by transition ───────────────────────────────
TOP_N = 10
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (stage, t_info) in zip(axes, TRANSITIONS.items()):
    delta_all = []
    for grp in GROUP_CONFIG:
        case_list = all_cf_results.get(grp, {}).get(stage, [])
        for case in case_list:
            delta_all.append(case["delta_df"].mean())

    if not delta_all:
        ax.set_title(f"{t_info['label']} — No data")
        continue

    delta_mean = pd.concat(delta_all, axis=1).mean(axis=1)
    top_idx    = delta_mean.abs().nlargest(TOP_N).index
    top_vals   = delta_mean[top_idx]

    bar_colors = ["#E57373" if v > 0 else "#64B5F6" for v in top_vals]
    bars = ax.barh(top_idx, top_vals, color=bar_colors, alpha=0.85,
                   edgecolor="white", linewidth=0.6)

    for bar, val in zip(bars, top_vals):
        ax.text(val + (0.003 if val >= 0 else -0.003),
                bar.get_y() + bar.get_height()/2,
                f"{val:+.3f}", va="center", fontsize=8.5,
                ha="left" if val >= 0 else "right")

    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{t_info['label']}\nMean feature change (top {TOP_N})",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Feature change magnitude (original scale)", fontsize=10)
    ax.invert_yaxis()
    ax.spines[["top","right"]].set_visible(False)

    # Legend
    ax.legend(handles=[
        mpatches.Patch(color="#E57373", alpha=0.85, label="Increase"),
        mpatches.Patch(color="#64B5F6", alpha=0.85, label="Decrease"),
    ], fontsize=9, loc="lower right")

plt.suptitle("DiCE Counterfactual — Mean Feature Changes by Transition Pathway",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_dice_feature_changes.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Feature change figure saved.")


2026-04-27 15:15:12,583 | INFO | Feature change figure saved.


In [8]:
# ── Figure: DiCE quality radar by group ──────────────────────────────────────
if len(quality_df) == 0:
    log.warning("No quality records — skipping radar chart")
else:
    categories    = ["Proximity\n(inverted)", "Diversity",
                     "Feasibility", "Sparsity\n(inverted)"]
    N             = len(categories)
    angles        = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                              subplot_kw=dict(polar=True))
    axes = axes.flatten()

    for ax, (grp, cfg) in zip(axes, GROUP_CONFIG.items()):
        sub = quality_df[quality_df["Group"] == GROUP_LABELS[grp]]
        color = GROUP_COLORS[grp]
        if len(sub) == 0:
            ax.set_title(GROUP_LABELS[grp], fontsize=10)
            ax.axis("off")
            continue

        prox_raw = sub["Proximity"].mean()
        # Invert proximity and sparsity: lower raw → higher radar score
        s_prox = max(0.0, 1.0 - min(prox_raw, 1.0))
        s_div  = min(sub["Diversity"].mean(), 1.0)
        s_feas = sub["Feasibility"].mean()
        s_spar = max(0.0, 1.0 - sub["Sparsity"].mean())
        scores = [s_prox, s_div, s_feas, s_spar]
        scores_closed = scores + [scores[0]]

        ax.plot(angles, scores_closed, "o-", color=color, lw=2.2, ms=6)
        ax.fill(angles, scores_closed, color=color, alpha=0.20)
        ax.plot(angles, [0.7]*5, "--", color="gray", lw=0.8, alpha=0.5)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, fontsize=8.5)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"],
                            fontsize=7, color="gray")

        for angle, score in zip(angles[:-1], scores):
            ax.text(angle, score + 0.09, f"{score:.2f}",
                    ha="center", fontsize=8.5, color=color, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white",
                              ec=color, alpha=0.8))
        ax.set_title(GROUP_LABELS[grp], fontsize=10, fontweight="bold", pad=10)

    plt.suptitle("DiCE Counterfactual Quality by Demographic Group\n"
                 "(higher score = better on each axis)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "fig_dice_quality_radar.png", dpi=DPI, bbox_inches="tight")
    plt.show()
    log.info("Quality radar saved.")


2026-04-27 15:15:14,734 | INFO | Quality radar saved.


In [9]:
# ── Figure: FPG transition flow (original → counterfactual) ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey=False)
axes      = axes.flatten()

stage_markers = {"diabetes": "o", "ifg": "s"}
stage_labels  = {"diabetes": "Diabetes→IFG", "ifg": "IFG→Normal"}

for ax, (grp, cfg) in zip(axes, GROUP_CONFIG.items()):
    color   = GROUP_COLORS[grp]
    plotted = False

    for stage, marker in stage_markers.items():
        case_list = all_cf_results.get(grp, {}).get(stage, [])
        for case in case_list:
            orig  = case["orig_fpg"]
            preds = case["pred_fpg"]
            for pred in preds:
                ax.annotate("",
                    xy=(1, pred), xytext=(0, orig),
                    arrowprops=dict(arrowstyle="->", color=color,
                                    lw=1.4, alpha=0.65))
                ax.scatter([0], [orig], color="gray",    s=45, zorder=5)
                ax.scatter([1], [pred], color=color, s=45,
                           marker=marker, zorder=5, edgecolors="white", lw=0.5)
                plotted = True

    if plotted:
        ax.axhline(100, color="#FF9800", linestyle="--",
                   lw=1.2, alpha=0.75, label="IFG boundary (100)")
        ax.axhline(126, color="#F44336", linestyle="--",
                   lw=1.2, alpha=0.75, label="Diabetes boundary (126)")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Original FPG", "Counterfactual FPG"], fontsize=9)
        ax.set_ylabel("Fasting Plasma Glucose (mg/dL)", fontsize=9)
        ax.legend(fontsize=7.5, loc="upper right")

    ax.set_title(GROUP_LABELS[grp], fontsize=10, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)

# Shared marker legend
legend_handles = [
    mpatches.Patch(color="gray",    label="● Original"),
    mpatches.Patch(color="#888888", label="● Diabetes → IFG"),
    mpatches.Patch(color="#888888", label="■ IFG → Normal"),
]
fig.legend(
    handles=[
        plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="gray",
                   markersize=9, label="Original FPG"),
        plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="#555",
                   markersize=9, label="CF: Diabetes→IFG"),
        plt.Line2D([0],[0], marker="s", color="w", markerfacecolor="#555",
                   markersize=9, label="CF: IFG→Normal"),
    ],
    loc="lower center", ncol=3, fontsize=9,
    bbox_to_anchor=(0.5, -0.03),
)
plt.suptitle("DiCE Counterfactual — FPG Stage Transition Flow by Group",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_dice_fpg_transition.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("FPG transition figure saved.")


2026-04-27 15:15:18,464 | INFO | FPG transition figure saved.


In [10]:
# ── Figure: Proximity by group (key fairness insight) ────────────────────────
# Shows that elderly groups require much larger feature changes → structural inequity

fig, ax = plt.subplots(figsize=(10, 5))

df_prox = (quality_df[quality_df["Transition"] == "IFG → Normal"]
           .groupby("Group")["Proximity"]
           .agg(["mean","std"])
           .reset_index())

# Sort by mean proximity descending
df_prox = df_prox.sort_values("mean", ascending=True)

bar_colors = [GROUP_COLORS.get(
    {v:k for k,v in GROUP_LABELS.items()}.get(g, ""), "#888888")
    for g in df_prox["Group"]]

bars = ax.barh(df_prox["Group"], df_prox["mean"], color=bar_colors,
               alpha=0.85, edgecolor="white", linewidth=0.6,
               xerr=df_prox["std"], capsize=4, error_kw={"ecolor":"gray","lw":1.2})

for bar, val in zip(bars, df_prox["mean"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9.5)

ax.set_xlabel("Mean Proximity (IFG → Normal transition)\n"
              "(lower = smaller lifestyle change needed)",
              fontsize=10)
ax.set_title("Counterfactual Difficulty by Demographic Group\n"
             "(Elderly groups require 2–10× larger changes than Young Male)",
             fontsize=11, fontweight="bold")
ax.axvline(df_prox["mean"].mean(), color="#C62828", linestyle="--",
           lw=1.5, alpha=0.7, label=f"Mean = {df_prox['mean'].mean():.3f}")
ax.legend(fontsize=9)
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_dice_proximity_comparison.png",
            dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Proximity comparison figure saved.")


2026-04-27 15:15:18,942 | INFO | Proximity comparison figure saved.


## 8. Summary & Outputs

In [11]:
log.info("=" * 55)
log.info("Notebook 03 complete.")
log.info("=" * 55)

outputs = [
    ("dice_results.pkl",            "All CF results — loaded by Notebook 04"),
    ("dice_quality_metrics.csv",    "Per-case quality metrics (Proximity/Diversity/Feasibility/Sparsity)"),
    ("dice_llm_input.csv",          "Structured LLM prompt input — loaded by Notebook 04"),
    ("fig_dice_feature_changes.png","Figure: Mean feature changes by transition"),
    ("fig_dice_quality_radar.png",  "Figure: Quality radar by group"),
    ("fig_dice_fpg_transition.png", "Figure: FPG stage transition flow"),
    ("fig_dice_proximity_comparison.png","Figure: Proximity comparison (fairness insight)"),
]
for fname, desc in outputs:
    log.info("  %-42s %s", fname, desc)

print("\n── Final Quality Summary ──")
print(quality_df.groupby("Transition")[
    ["Proximity","Diversity","Feasibility","Sparsity"]
].mean().round(4).to_string())


2026-04-27 15:15:18,971 | INFO | =======================================================
2026-04-27 15:15:18,976 | INFO | Notebook 03 complete.
2026-04-27 15:15:18,980 | INFO | =======================================================
2026-04-27 15:15:18,984 | INFO |   dice_results.pkl                           All CF results — loaded by Notebook 04
2026-04-27 15:15:18,985 | INFO |   dice_quality_metrics.csv                   Per-case quality metrics (Proximity/Diversity/Feasibility/Sparsity)
2026-04-27 15:15:18,985 | INFO |   dice_llm_input.csv                         Structured LLM prompt input — loaded by Notebook 04
2026-04-27 15:15:18,986 | INFO |   fig_dice_feature_changes.png               Figure: Mean feature changes by transition
2026-04-27 15:15:18,988 | INFO |   fig_dice_quality_radar.png                 Figure: Quality radar by group
2026-04-27 15:15:18,989 | INFO |   fig_dice_fpg_transition.png                Figure: FPG stage transition flow
2026-04-27 15:15:18,990 | INFO |


── Final Quality Summary ──
                Proximity  Diversity  Feasibility  Sparsity
Transition                                                 
Diabetes → IFG     0.1149     0.1937       0.9737    0.0532
IFG → Normal       0.3620     0.2773       0.9259    0.1493
